In [ ]:
!pip install langchain youtube-transcript-api google-genai tiktoken transformers faiss-cpu langchain-community python-dotenv

: 

# Imports

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os
load_dotenv()

: 

# Fetching transcripts from video via video ID

In [ ]:
video_id = "LPZh9BOjkQs"

try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages = ['en'])    # fetches transcript along with lots of metadata
    transcript_text = " ".join(chunk.text for chunk in transcript_list) # extracts only the text content
except TranscriptsDisabled:
    print("Transcript not available for this video.")

print(transcript_text)

: 

# Adding Text Splitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.split_text(transcript_text)

# Generate embeddings and store in vector DB

In [ ]:
embedder = GoogleGenerativeAIEmbeddings(model = "models/gemini-embedding-001")

vector_db = FAISS.from_texts(chunks, embedder)

# Basic retriever

In [ ]:
retriever = vector_db.as_retriever(search_type = "similarity", search_kwargs = {"k" : 3})

In [ ]:
query = "AI models used in text generation?"

results = retriever.invoke(query)
context_text = " ".join(doc.page_content for doc in results)
context_text

"it does is assign a probability to all possible next words. To build a chatbot, you lay out some text that describes an interaction between a user and a hypothetical AI assistant, add on whatever the user types in as the first part of the interaction, and then have the model repeatedly predict the next word that such a hypothetical AI assistant would say in response, and that's what's presented to the user. In doing this, the output tends to look a lot more natural if you allow it to select less likely words along the way at random. So what this means is even though the model itself is deterministic, a given prompt typically gives a different answer each time it's run. Models learn how to make these predictions by processing an enormous amount of text, typically pulled from the internet. For a standard human to read the amount of text that was used to train GPT-3, for example, if they read non-stop 24-7, it would take over 2600 years. Larger models since then train on much, much more.

# Prompt template

In [ ]:
prompt = PromptTemplate(
    template = """
    You are an helpful AI assistant. Use the following context to answer the question.
    Context: {context}
    Question : {question}
""",
    input_variables = ["context", "question"]
)

# Including LLM

In [ ]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

final_prompt = prompt.invoke({"context" : context_text, "question" : query})

response = llm.invoke(final_prompt)
print(response.content)

AI models used in text generation are referred to as **large language models**.

These models are sophisticated mathematical functions that predict what word comes next for any piece of text. They do this by assigning a probability to all possible next words. They learn by processing an enormous amount of text, typically pulled from the internet, with models like GPT-3 being mentioned as an example.


# Building the above RAG again, but using chains

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
    context_text = "\n \n ".join(doc.page_content for doc in retrieved_docs)
    return context_text

## Parallel chain

In [ ]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke("Who is Demise")